# Thread 4: Suppression of Dissent

### This thread looks at the different instances when the government unfairly used it's power to silence the population with conflicting opinion challenging it's authority.

## EDA 

# Importing Libraries

In [13]:
import re
import pdfplumber
import pandas as pd
from docx import Document
import sys
sys.path.append("..")
import utility as u
import importlib
importlib.reload(u)

<module 'utility' from '/Users/bhavya/Documents/DS/The Committee Will Look Into It/04_Suppression_of_Dissent/../utility.py'>

In [14]:
with pdfplumber.open("data/01_144-Report-release-version.pdf") as pdf:
    for page in pdf.pages:
        table = page.extract_table()
        if table:
            print(table)


[['In our personal experience, every ACP office in Delhi maintained a dedicated file\nfor Section 144 orders issued each year. Despite this, in many cases, we were not\nstraight away provided with a copy of all the orders (and had to inspect files) on\nthe ground that doing so required a huge diversion of (rather limited) personnel.', '']]
[['', '', '“larger public interest”\nThe South District refused to supply us with copies of orders issued\nunder Section 144, CrPC citing Section 8(1)(e) of the RTI Act –\nsomehow concluding that the “larger public interest” did not warrant\ndisclosure of orders statutorily obligated to be made publicly available.\nNevertheless, they permitted physical inspection of over 400 orders']]
[['Ranges', 'Districts', 'Sub-divisions'], ['Eastern Range', 'North- East District', '1. Seelampur\n2. Gokul Puri\n3. Khajuri Khas\n4. Nand Nagri\n5. Bhajan Pura'], [None, 'Shahdara District', '1. Shahdara\n2. Vivek Vihar\n3. Gandhinagar\n4. Seema Puri'], [None, 'East D

In [15]:
doc = Document("data/02_Internet shutdown.docx")
for para in doc.paragraphs[:30]:
    print(para.text)

for table in doc.tables:
    for row in table.rows:
        print([cell.text for cell in row.cells])

Internet shutdown










['State', 'District(s)', 'Network', 'Nature', 'Published Date', 'Status', 'Source']
['Bihar', 'Muzaffarpur', 'Mobile', 'Preventive', '24 Jul 2026', 'Ended', 'Order PDF']
['NCT of Delhi', 'Central Delhi', 'Mobile', 'Preventive', '24 Jul 2026', 'Ended', 'Order PDF']
['NCT of Delhi', 'Central Delhi', 'Mobile', 'Preventive', '23 Jul 2026', 'Ended', 'Order PDF']
['NCT of Delhi', 'Central Delhi', 'Na', 'Reactive', '22 Jul 2026', 'Ended', 'Order PDF']
['NCT of Delhi', 'Central Delhi', 'Mobile', 'Reactive', '20 Jul 2026', 'Ended', 'Order PDF']
['Jammu and Kashmir', 'Doda', 'Na', 'Preventive', '17 Jul 2026', 'Ended', 'Order PDF']
['Rajasthan', 'Karauli, Sawai Madhopur', 'Mobile', 'Preventive', '08 Jul 2026', 'Ended', 'Order PDF']
['Uttrakhand', 'Chamoli', 'Both', 'Preventive', '21 Jun 2026', 'Ended', 'Order PDF']
['Rajasthan', 'Jaipur', 'Mobile', 'Preventive', '07 Jun 2026', 'Ended', 'Order PDF']
['Haryana', 'Faridabad', 'Mobile', 'Preventive', '30 May 2026', 'Ended'

In [16]:
rows = []

with pdfplumber.open("data/01_144-Report-release-version.pdf") as pdf:
    for page in pdf.pages:
        tables = page.extract_tables()
        for table in tables:
            for row in table:
                # rows that have a district number and total
                if row[0] and row[0].strip().replace('.','').isdigit() and row[-1]:
                    district = row[1].strip() if row[1] else None
                    total = row[-1].strip() if row[-1] else None
                    if district and total and total.isdigit():
                        rows.append({
                            "district": district,
                            "sec144_order_count": int(total)
                        })

df_pdf = pd.DataFrame(rows)
df_pdf["state"] = "Delhi"
df_pdf["event_type"] = "sec144"
df_pdf["district"] = df_pdf["district"].str.replace(r'\d+$', '', regex=True).str.strip()

print(df_pdf)
df_pdf.to_csv("data/file1.csv", index=False)

                           district  sec144_order_count  state event_type
0                  Central District                  28  Delhi     sec144
1                   Dwarka District                 450  Delhi     sec144
2                     East District                 508  Delhi     sec144
3                New Delhi District                 383  Delhi     sec144
4                    North District                 520  Delhi     sec144
5               North-East District                 541  Delhi     sec144
6               North-West District                 342  Delhi     sec144
7                    Outer District                 431  Delhi     sec144
8              Outer-North District                 394  Delhi     sec144
9                   Rohini District                 409  Delhi     sec144
10                Shahdara District                 578  Delhi     sec144
11                   South District                 456  Delhi     sec144
12              South-East District   

In [17]:
doc = Document("data/02_Internet shutdown.docx")

rows = []
for table in doc.tables:
    for row in table.rows:
        cells = [cell.text.strip() for cell in row.cells]
        # skip header rows
        if cells[0] == "State":
            continue
        if len(cells) >= 6 and cells[0]:
            rows.append({
                "state":        cells[0],
                "district":     cells[1],
                "network_type": cells[2] if cells[2] != "Na" else None,
                "nature":       cells[3],
                "date":         cells[4],
                "status":       cells[5]
            })

df_docx = pd.DataFrame(rows)
df_docx["event_type"] = "internet_shutdown"

# fix date
df_docx["date"] = pd.to_datetime(df_docx["date"], format="%d %b %Y", errors="coerce")

# fix malformed district strings like "['Kakching', 'Imphal West']"
def clean_district(val):
    if val.startswith("["):
        val = re.sub(r"[\[\]']", "", val)
    return val.strip()

df_docx["district"] = df_docx["district"].apply(clean_district)

# standardize state names
df_docx["state"] = df_docx["state"].str.strip().str.title()
df_docx["state"] = df_docx["state"].replace({
    "Nct Of Delhi": "Delhi",
    "Jammu & Kashmir": "Jammu And Kashmir",
    "Uttrakhand": "Uttarakhand"
})

print(df_docx.head(10))
print(df_docx.shape)
df_docx.to_csv("data/file2.csv", index=False)

               state                 district network_type      nature  \
0              Bihar              Muzaffarpur       Mobile  Preventive   
1              Delhi            Central Delhi       Mobile  Preventive   
2              Delhi            Central Delhi       Mobile  Preventive   
3              Delhi            Central Delhi         None    Reactive   
4              Delhi            Central Delhi       Mobile    Reactive   
5  Jammu And Kashmir                     Doda         None  Preventive   
6          Rajasthan  Karauli, Sawai Madhopur       Mobile  Preventive   
7        Uttarakhand                  Chamoli         Both  Preventive   
8          Rajasthan                   Jaipur       Mobile  Preventive   
9            Haryana                Faridabad       Mobile  Preventive   

        date status         event_type  
0 2026-07-24  Ended  internet_shutdown  
1 2026-07-24  Ended  internet_shutdown  
2 2026-07-23  Ended  internet_shutdown  
3 2026-07-22  Ended  

In [18]:
csv1= u.load_data("data/file1.csv")
csv2 = u.load_data("data/file2.csv")
csv2["date"] = pd.to_datetime(df_docx["date"], format="%Y-%m-%d")


csv1 = csv1[["state", "district", "event_type", "sec144_order_count"]]
csv2 = csv2[["state", "district", "network_type", "nature", "date", "status", "event_type"]]


df = pd.concat([csv1, csv2], ignore_index=True)
df.to_csv("data/df_combined.csv", index=False)
print(df.shape)
print(df.dtypes)

(120, 8)
state                         object
district                      object
event_type                    object
sec144_order_count           float64
network_type                  object
nature                        object
date                  datetime64[ns]
status                        object
dtype: object


In [19]:
df["sec144_order_count"] = df["sec144_order_count"].astype("Int64")  # capital I — nullable int

In [20]:
df = df.drop_duplicates()

In [21]:
print(sorted(df["state"].unique()))

['Assam', 'Bihar', 'Delhi', 'Haryana', 'Jammu And Kashmir', 'Ladakh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Odisha', 'Punjab', 'Rajasthan', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']


In [22]:
u.eda(df)

=== DataFrame ===
Shape       : 119 rows × 8 cols
Duplicates  : 0

Column Summary:
                             Dtype  Nulls  Null %  Uniques    Flag
state                       object      0    0.00       17        
district                    object      0    0.00       83        
event_type                  object      0    0.00        2        
sec144_order_count           Int64     99   83.19       19  SPARSE
network_type                object     38   31.93        3        
nature                      object     20   16.81        2        
date                datetime64[ns]     20   16.81       88        
status                      object     20   16.81        1        

Sparse Columns (>50% null): ['sec144_order_count']

Numeric Stats:
                    min    max   mean  median
sec144_order_count  2.0  578.0  306.2   376.5

Sample:
   state          district event_type  sec144_order_count network_type nature date status
0  Delhi  Central District     sec144                  

In [23]:
df= u.fill_numeric_nulls(df, ['sec144_order_count', 'network_type', 'nature', 'status'])

df["sec144_order_count"] = df["sec144_order_count"].replace(0, pd.NA)
df["network_type"] = df["network_type"].replace("0", pd.NA)
df["nature"] = df["nature"].replace("0", pd.NA)
df["status"] = df["status"].replace("0", pd.NA)

df["district"] = df["district"].replace("-", pd.NA)

df["network_type"] = df["network_type"].replace("Mobile Data", "Mobile")

u.save_clean(df, "data/df_cleaned.csv")

Saved | shape- (119, 8)


In [24]:
u.eda(df)

=== DataFrame ===
Shape       : 119 rows × 8 cols
Duplicates  : 0

Column Summary:
                             Dtype  Nulls  Null %  Uniques    Flag
state                       object      0    0.00       17        
district                    object      1    0.84       82        
event_type                  object      0    0.00        2        
sec144_order_count           Int64     99   83.19       19  SPARSE
network_type                object      0    0.00        3        
nature                      object      0    0.00        3        
date                datetime64[ns]     20   16.81       88        
status                      object      0    0.00        2        

Sparse Columns (>50% null): ['sec144_order_count']

Numeric Stats:
                    min    max   mean  median
sec144_order_count  2.0  578.0  306.2   376.5

Sample:
   state          district event_type  sec144_order_count network_type nature date status
0  Delhi  Central District     sec144                  